# SL-3 — Apprentissage basé sur la pertinence (twin C# from-scratch)

Ce notebook est le **jumeau C# / .NET** de [SL-3-RelevanceLearning.ipynb](SL-3-RelevanceLearning.ipynb) — marathon de parité #4956.

La version Python déroule le **Relevance-Based Learning (RBL, AIMA §19.4)** en pur Python. Ce twin C# **réimplémente le moteur
from-scratch** (BCL .NET 9, 0 NuGet) : déterminations fonctionnelles, treillis des déterminations, clé minimale,
borne PAC, information mutuelle pour la sélection d'attributs. sklearn/Python = `RECOVERABLE-MACHINE` ; le moteur from-scratch EST la substance.

**Idée centrale (RBL)** : au lieu d'apprendre dans l'espace d'hypothèses complet (2^n sous-ensembles d'attributs),
on cherche le **plus petit sous-ensemble d'attributs qui détermine fonctionnellement la cible** — la *détermination minimale*.
Cela réduit exponentiellement l'espace (2^n → 2^d, d ≪ n) et le nombre d'exemples requis devient linéaire en d.

## 1. Déterminations fonctionnelles

Une **détermination** `attrs → target` tient si, pour tout groupe de lignes partageant les mêmes valeurs d'`attrs`,
la valeur de `target` est constante. C'est l'analogue d'une **dépendance fonctionnelle** en théorie des bases de données.
Une violation = deux lignes mêmes attrs, target différent.

In [1]:
using System;
using System.Linq;
using System.Text;
using System.Collections.Generic;

static void Show(string s) { s.Display(); }
Show("Apprentissage base sur la pertinence (RBL, AIMA 19.4) — moteur from-scratch.");


Apprentissage base sur la pertinence (RBL, AIMA 19.4) — moteur from-scratch.

In [2]:
#nullable enable
// Une ligne de donnees : dictionnaire attribut -> valeur (string, domaine discret).
public sealed class Row : Dictionary<string,string> { }

// Resultat de la verification d'une determination, avec diagnostic des violations.
public sealed class DeterminationResult
{
    public List<string> Attrs = new();
    public string Target = "";
    public bool Holds;
    public List<(Row A, Row B)> Violations = new();  // paires de lignes en conflit
    public int Coverage;                              // nombre de groupes distincts
}

// Verifie si detAttrs determinent target (target constante dans chaque groupe).
public static DeterminationResult CheckDetermination(List<Row> data, List<string> detAttrs, string targetAttr)
{
    var res = new DeterminationResult { Attrs = detAttrs, Target = targetAttr };
    // Grouper par signature des attrs determinants.
    var groups = new Dictionary<string, List<Row>>();
    foreach (var row in data)
    {
        string key = string.Join("\x1f", detAttrs.Select(a => row[a]));  // unit separator evite les collisions
        if (!groups.ContainsKey(key)) groups[key] = new List<Row>();
        groups[key].Add(row);
    }
    res.Coverage = groups.Count;
    foreach (var kv in groups)
    {
        var targetValues = kv.Value.Select(r => r[targetAttr]).Distinct().ToList();
        if (targetValues.Count > 1)
        {
            // Violation : on garde la premiere paire en conflit pour le diagnostic.
            res.Violations.Add((kv.Value[0], kv.Value[1]));
        }
    }
    res.Holds = res.Violations.Count == 0;
    return res;
}

Show("CheckDetermination pret : groupe par signature, detecte les violations de target constante.");


CheckDetermination pret : groupe par signature, detecte les violations de target constante.

## 2. Treillis des déterminations (powerset)

L'ensemble des sous-ensembles d'attributs forme un **treillis**. Pour chaque sous-ensemble, on note s'il détermine la cible.
Visualiser ce treillis révèle la structure : la détermination est **anti-monotone** (si un ensemble détermine, ses sur-ensembles aussi).

In [3]:
#nullable enable

// Toutes les combinaisons de taille k (substitut a itertools.combinations).
public static IEnumerable<List<string>> Combinations(List<string> items, int k)
{
    if (k == 0) { yield return new List<string>(); yield break; }
    if (k > items.Count) yield break;
    for (int i = 0; i <= items.Count - k; i++)
    {
        var head = items[i];
        foreach (var tail in Combinations(items.Skip(i + 1).ToList(), k - 1))
        {
            var combo = new List<string> { head };
            combo.AddRange(tail);
            yield return combo;
        }
    }
}

// Cle trieee pour indexer le lattice (evite HashSet comme cle de dict).
public static string SortedKey(List<string> attrs) => string.Join(",", attrs.OrderBy(a => a));

// Construit le treillis complet : chaque sous-ensemble -> tient ou non.
public static Dictionary<string,bool> BuildDeterminationLattice(List<Row> data, List<string> allAttrs, string targetAttr)
{
    var lattice = new Dictionary<string,bool>();
    for (int size = 0; size <= allAttrs.Count; size++)
        foreach (var subset in Combinations(allAttrs, size))
        {
            bool holds;
            if (size == 0)
            {
                // L'ensemble vide ne determine que s'il n'y a qu'une seule valeur cible.
                holds = data.Select(r => r[targetAttr]).Distinct().Count() == 1;
            }
            else
            {
                holds = CheckDetermination(data, subset, targetAttr).Holds;
            }
            lattice[SortedKey(subset)] = holds;
        }
    return lattice;
}

Show("BuildDeterminationLattice pret : powerset complet evalue.");


BuildDeterminationLattice pret : powerset complet evalue.

## 3. Détermination minimale (clé candidate)

La **détermination minimale** est le plus petit sous-ensemble qui détermine la cible.
On l'obtient par recherche en largeur (BFS par taille croissante) — le premier trouvé est minimal.
C'est l'analogue d'une **clé candidate minimale** en bases de données.

In [4]:
#nullable enable

public sealed class DetSearchResult
{
    public List<string>? Determination;
    public int SubsetsTested;
    public int LevelsExplored;
    public int? FoundAtLevel;
}

// Recherche BFS par taille croissante : retourne le plus petit sous-ensemble valide.
public static DetSearchResult MinimalConsistentDet(List<Row> data, List<string> allAttrs, string targetAttr, bool verbose = true)
{
    int totalTested = 0;
    var sb = new StringBuilder();
    for (int size = 1; size <= allAttrs.Count; size++)
    {
        var subsets = Combinations(allAttrs, size).ToList();
        if (verbose) sb.AppendLine($"  Niveau {size} : test de {subsets.Count} combinaisons...");
        foreach (var subset in subsets)
        {
            totalTested++;
            if (CheckDetermination(data, subset, targetAttr).Holds)
            {
                if (verbose) sb.AppendLine($"    TROUVE : {{{string.Join(", ", subset)}}} apres {totalTested} tests");
                if (verbose) Show(sb.ToString());
                return new DetSearchResult { Determination = subset, SubsetsTested = totalTested, LevelsExplored = size, FoundAtLevel = size };
            }
        }
    }
    if (verbose) { sb.AppendLine($"  Aucune determination trouvee ({totalTested} tests)"); Show(sb.ToString()); }
    return new DetSearchResult { Determination = null, SubsetsTested = totalTested, LevelsExplored = allAttrs.Count, FoundAtLevel = null };
}

Show("MinimalConsistentDet pret : BFS par taille, retourne le + petit sous-ensemble valide.");


MinimalConsistentDet pret : BFS par taille, retourne le + petit sous-ensemble valide.

## 4. Application — activité biologique des molécules

Quelles propriétés moléculaires (mw, logp, hbd, hba) déterminent si une molécule est **active** ?
La recherche de détermination minimale répond en explorant le treillis par taille croissante.

In [5]:
#nullable enable

Row R(params (string k,string v)[] pairs)
{
    var d = new Row();
    foreach (var (k,v) in pairs) d[k] = v;
    return d;
}

var moleculeData = new List<Row>
{
    R(("mw","low"),("logp","low"),("hbd","low"),("hba","low"),("activity","inactive")),
    R(("mw","low"),("logp","low"),("hbd","low"),("hba","high"),("activity","inactive")),
    R(("mw","medium"),("logp","high"),("hbd","low"),("hba","high"),("activity","active")),
    R(("mw","medium"),("logp","high"),("hbd","high"),("hba","high"),("activity","active")),
    R(("mw","high"),("logp","high"),("hbd","low"),("hba","high"),("activity","active")),
    R(("mw","high"),("logp","high"),("hbd","high"),("hba","high"),("activity","active")),
    R(("mw","high"),("logp","low"),("hbd","high"),("hba","low"),("activity","inactive")),
    R(("mw","medium"),("logp","low"),("hbd","high"),("hba","low"),("activity","inactive")),
    R(("mw","low"),("logp","high"),("hbd","low"),("hba","low"),("activity","inactive")),
    R(("mw","medium"),("logp","high"),("hbd","low"),("hba","low"),("activity","active")),
};

var MOL_ATTRS = new List<string> { "mw", "logp", "hbd", "hba" };

Show("Donnees : proprietes moleculaires (10 molecules)");
var hdr = new StringBuilder();
hdr.AppendLine($"{"MW",8} | {"LogP",6} | {"HBD",5} | {"HBA",6} | {"Activite",10}");
hdr.AppendLine(new string('-', 50));
foreach (var row in moleculeData)
    hdr.AppendLine($"{row["mw"],8} | {row["logp"],6} | {row["hbd"],5} | {row["hba"],6} | {row["activity"],10}");
Show(hdr.ToString());

Show("\nRecherche de determination minimale :");
var molRes = MinimalConsistentDet(moleculeData, MOL_ATTRS, "activity");

if (molRes.Determination is not null)
{
    int d = molRes.Determination.Count, n = MOL_ATTRS.Count;
    Show($"\nDetermination minimale : {{{string.Join(", ", molRes.Determination)}}}");
    Show($"Sous-ensembles testes : {molRes.SubsetsTested} / {(1 << n) - 1}");
    Show($"Reduction espace : O(2^{n}) -> O(2^{d}), facteur {1 << (n - d)}");
}
else
{
    Show("\nAucune determination fonctionnelle trouvee.");
    Show("-> Le concept est probablement disjonctif ou depend d'interactions.");
}


Donnees : proprietes moleculaires (10 molecules)

      MW |   LogP |   HBD |    HBA |   Activite
--------------------------------------------------
     low |    low |   low |    low |   inactive
     low |    low |   low |   high |   inactive
  medium |   high |   low |   high |     active
  medium |   high |  high |   high |     active
    high |   high |   low |   high |     active
    high |   high |  high |   high |     active
    high |    low |  high |    low |   inactive
  medium |    low |  high |    low |   inactive
     low |   high |   low |    low |   inactive
  medium |   high |   low |    low |     active



Recherche de determination minimale :

  Niveau 1 : test de 4 combinaisons...
  Niveau 2 : test de 6 combinaisons...
    TROUVE : {mw, logp} apres 5 tests



Determination minimale : {mw, logp}

Sous-ensembles testes : 5 / 15

Reduction espace : O(2^4) -> O(2^2), facteur 4

## 5. Borne PAC — l'avantage quantifié

En apprentissage PAC (Probably Approximately Correct), le nombre d'exemples suffisant pour apprendre dans un espace d'hypothèses fini vaut :

> m ≥ (1/ε)·(ln|H| + ln(1/δ)),  avec |H| = 2^n (sélection d'attributs).

Le RBL réduit |H| de 2^n à 2^d (d = taille de la détermination) : le nombre d'exemples devient **linéaire en d** au lieu de n.

In [6]:
#nullable enable

// Borne PAC : m >= (1/eps) * (ln|H| + ln(1/delta)), avec |H| = 2^n_attrs.
public static int PacBound(int nAttrs, double epsilon = 0.1, double delta = 0.05)
{
    double lnH = nAttrs * Math.Log(2);          // ln(|H|) = n * ln(2)
    return (int)Math.Ceiling((1.0 / epsilon) * (lnH + Math.Log(1.0 / delta)));
}

Show("Analyse de complexite : RBL vs selection aveugle\n");
// |H| = 2^k ; pour k grand (> 62) le cast long deborde : on affiche alors la forme symbolique.
string HStr(int k) => k <= 62 ? ((long)Math.Pow(2, k)).ToString() : $"2^{k}";

Show($"{"n (total)",10} | {"d (pertinent)",14} | {"|H| sans RBL",14} | {"|H| avec RBL",14} | {"m sans",7} | {"m avec",7}");
Show(new string('-', 78));
foreach (var (n, d) in new[] { (5,1), (10,2), (20,2), (20,3), (50,3), (100,5) })
{
    Show($"{n,10} | {d,14} | {HStr(n),14} | {HStr(d),14} | {PacBound(n),7} | {PacBound(d),7}");
}
Show("\nLa reduction de l'espace est EXPONENTIELLE (2^n -> 2^d) ; m devient lineaire en d.");
Show($"Pour n=100 et d=5 : {PacBound(100)} exemples sans RBL contre {PacBound(5)} avec la determination.");


Analyse de complexite : RBL vs selection aveugle


 n (total) |  d (pertinent) |   |H| sans RBL |   |H| avec RBL |  m sans |  m avec

------------------------------------------------------------------------------

         5 |              1 |             32 |              2 |      65 |      37

        10 |              2 |           1024 |              4 |     100 |      44

...

        20 |              3 |        1048576 |              8 |     169 |      51

        50 |              3 | 1125899906842624 |              8 |     377 |      51

       100 |              5 |          2^100 |             32 |     724 |      65


La reduction de l'espace est EXPONENTIELLE (2^n -> 2^d) ; m devient lineaire en d.

Pour n=100 et d=5 : 724 exemples sans RBL contre 65 avec la determination.

## 6. Parallèle RBL ↔ Web Sémantique (OWL)

Une détermination `metal → conductivité` (RBL) correspond à déclarer `:aConductivite` comme **`owl:FunctionalProperty`** :
chaque sujet a au plus une valeur. Un reasoner OWL (HermiT, Pellet) détecterait les violations comme une incohérence.

In [7]:
#nullable enable

// Mini triple-store (sujet, predicat, objet) en pur C#.
public sealed record Triple(string S, string P, string O);

// Detecte les violations de fonctionnalite d'une propriete OWL :
// un sujet avec plusieurs objets distincts = incoherence.
public static Dictionary<string, List<string>> FunctionalViolations(List<Triple> triples, string prop)
{
    var objects = new Dictionary<string, HashSet<string>>();
    foreach (var t in triples)
        if (t.P == prop)
        {
            if (!objects.ContainsKey(t.S)) objects[t.S] = new HashSet<string>();
            objects[t.S].Add(t.O);
        }
    return objects.Where(kv => kv.Value.Count > 1)
                  .OrderBy(kv => kv.Key)
                  .ToDictionary(kv => kv.Key, kv => kv.Value.OrderBy(o => o).ToList());
}

var copperData = new List<Row>
{
    R(("metal","Cu"),("conductivity","high")),
    R(("metal","Ag"),("conductivity","high")),
    R(("metal","Fe"),("conductivity","medium")),
    R(("metal","Cu"),("conductivity","high")),   // doublon coherent
};

var triples = new List<Triple>();
foreach (var r in copperData) triples.Add(new Triple($":{r["metal"]}", ":aConductivite", r["conductivity"]));
triples.Add(new Triple(":aConductivite", "rdf:type", "owl:FunctionalProperty"));

// Injectons une incoherence : Fe avec 2 valeurs.
triples.Add(new Triple(":Fe", ":aConductivite", "high"));

var viol = FunctionalViolations(triples, ":aConductivite");
Show("Parallele RBL <-> Web Semantique : la MEME contrainte, deux formalismes");
Show($"Propriete fonctionnelle :aConductivite — violations detectees :");
if (viol.Count == 0) Show("  (aucune — coherent)");
foreach (var kv in viol)
    Show($"  {kv.Key} a {kv.Value.Count} objets distincts : [{string.Join(", ", kv.Value)}]");
Show("\nUne determination RBL = exactement une owl:FunctionalProperty ; les deux formalismes");
Show("expriment la meme contrainte fonctionnelle (un sujet -> au plus un objet).");


Parallele RBL <-> Web Semantique : la MEME contrainte, deux formalismes

Propriete fonctionnelle :aConductivite — violations detectees :

  :Fe a 2 objets distincts : [high, medium]


Une determination RBL = exactement une owl:FunctionalProperty ; les deux formalismes

expriment la meme contrainte fonctionnelle (un sujet -> au plus un objet).

## 7. Information mutuelle — sélection guidée

Le treillis des déterminations grandit comme 2^n. Une heuristique : guider la recherche par **l'information mutuelle**
I(attr ; target). Les attributs à forte MI sont des candidats prioritaires pour la détermination.

In [8]:
#nullable enable

// Entropie de Shannon (base 2) d'une distribution de valeurs.
public static double Entropy(IEnumerable<string> values)
{
    var list = values.ToList();
    if (list.Count == 0) return 0;
    return list.GroupBy(v => v)
               .Select(g => (double)g.Count() / list.Count)
               .Sum(p => -p * Math.Log2(p));
}

// Information mutuelle I(attr ; target) = H(target) - H(target | attr).
public static double MutualInformation(List<Row> data, string attr, string targetAttr)
{
    double hTarget = Entropy(data.Select(r => r[targetAttr]));
    double hCond = data.GroupBy(r => r[attr])
                      .Select(g => (double)g.Count() / data.Count * Entropy(g.Select(r => r[targetAttr])))
                      .Sum();
    return hTarget - hCond;
}

Show("Scores d'information mutuelle des proprietes moleculaires vs activite :");
var miScores = MOL_ATTRS.ToDictionary(a => a, a => MutualInformation(moleculeData, a, "activity"));
foreach (var kv in miScores.OrderByDescending(kv => kv.Value))
    Show($"  I({kv.Key} ; activity) = {kv.Value:F4} bits");

Show("\nL'attribut avec la plus forte MI est le meilleur candidat pour demarrer la recherche de determination.");


Scores d'information mutuelle des proprietes moleculaires vs activite :

  I(logp ; activity) = 0,6100 bits

  I(mw ; activity) = 0,4000 bits

  I(hba ; activity) = 0,2781 bits

  I(hbd ; activity) = 0,0000 bits


L'attribut avec la plus forte MI est le meilleur candidat pour demarrer la recherche de determination.

## 8. Exercices

### Exercice 1 — Prédiction par détermination

Utilisez la détermination minimale trouvée (section 4) pour **prédire** l'activité d'une nouvelle molécule.
Si ses valeurs pour les attributs déterminants correspondent à un groupe connu, renvoyez l'activité de ce groupe ; sinon `"?"`.

> Indice : groupez `moleculeData` par signature de la détermination ; cherchez le groupe correspondant à la nouvelle ligne.

In [9]:
#nullable enable

// TODO etudiant : predire l'activite d'une nouvelle molecule via la determination minimale.
// Si aucune ligne connue ne partage la signature -> renvoyer "?".
static string PredictActivity(List<Row> data, List<string> detAttrs, Row newRow)
{
    // Indice : grouper data par signature detAttrs, chercher le groupe de newRow.
    return "?";  // TODO etudiant
}

var testRow = R(("mw","high"),("logp","high"),("hbd","low"),("hba","high"));
string? predicted = null;  // TODO etudiant : PredictActivity(moleculeData, molRes.Determination ?? MOL_ATTRS, testRow);
Show("Exercice 1 a completer");
Show($"  (attendu pour {testRow["mw"]}/{testRow["logp"]}/{testRow["hbd"]}/{testRow["hba"]} : 'active')");


Exercice 1 a completer

  (attendu pour high/high/low/high : 'active')

### Exercice 2 — Borne PAC stratifiée

Après RBL, la borne PAC peut être **resserrée** : si la détermination a taille d, |H| = 2^d.
Implémentez une borne qui prend en compte la détermination trouvée et comparez-la à la borne brute.

In [10]:
#nullable enable

// TODO etudiant : borne PAC apres RBL (|H| = 2^d au lieu de 2^n).
// Retourner (mSansRBL, mAvecRBL) pour n attributs dont d pertinents.
static (int mSans, int mAvec) StratifiedPacBound(int n, int d, double epsilon = 0.1, double delta = 0.05)
{
    // Indice : reutiliser PacBound avec n puis d.
    return (0, 0);  // TODO etudiant
}

var sp = (0, 0);  // TODO etudiant : StratifiedPacBound(20, 2);
Show("Exercice 2 a completer");
Show("  (attendu : mSans > mAvec, reduction ~ factorielle en n-d)");


Exercice 2 a completer

  (attendu : mSans > mAvec, reduction ~ factorielle en n-d)

### Exercice 3 — Pruning anti-monotone du treillis

La détermination est **anti-monotone** : si un sous-ensemble ne détermine pas la cible, aucun de ses sous-ensembles ne le peut non plus
(en fait c'est l'inverse — si un sur-ensemble détermine, ses sous-ensembles aussi... à vérifier !).
Implémentez une recherche de détermination minimale qui **élague** le treillis en exploitant cette propriété pour éviter des tests inutiles.

In [11]:
#nullable enable

// TODO etudiant : MinimalConsistentDet avec pruning anti-monotone.
// Si un sous-ensemble de taille k ne determine PAS, aucun sous-ensemble de taille < k contenu dedans
// ne peut determiner (propriete a verifier/demontrer). Elager en consequence.
static List<string>? PrunedMinimalDet(List<Row> data, List<string> allAttrs, string targetAttr)
{
    // Indice : memoiser les sous-ensembles echoues ; skipper leurs subsets.
    return null;  // TODO etudiant
}

List<string>? pruned = null;  // TODO etudiant : PrunedMinimalDet(moleculeData, MOL_ATTRS, "activity");
Show("Exercice 3 a completer");
Show("  (reflechir : anti-monotonie = si A determine, tout sur-ensemble de A determine aussi)");


Exercice 3 a completer

  (reflechir : anti-monotonie = si A determine, tout sur-ensemble de A determine aussi)

---

## Synthèse

Ce twin C# a réimplémenté **from-scratch** les piliers du Relevance-Based Learning :

| Concept | Implémentation |
|---------|----------------|
| Détermination fonctionnelle | `CheckDetermination` (groupage par signature) |
| Treillis des déterminations | `BuildDeterminationLattice` (powerset) |
| Détermination minimale | `MinimalConsistentDet` (BFS par taille) |
| Borne PAC | `PacBound` (m ≥ (1/ε)·(ln2^n + ln(1/δ))) |
| Parallèle OWL | `FunctionalViolations` (owl:FunctionalProperty) |
| Sélection par MI | `MutualInformation` (H(target) − H(target|attr)) |

La lib Python (sklearn/AIMA) fournit ces constructions en boîte noire ; l'implémentation from-scratch rend **visible**
pourquoi la détermination minimale réduit exponentiellement l'espace d'hypothèses (2^n → 2^d) et rend le nombre d'exemples linéaire en d.

**Voir aussi** : [SL-3-RelevanceLearning.ipynb](SL-3-RelevanceLearning.ipynb) (Python), [SL-4-InductiveLogicProgramming-Csharp.ipynb](SL-4-InductiveLogicProgramming-Csharp.ipynb), marathon #4956, EPIC #3801.